# DO Within-1-Day Temporal Model Comparison

Goal: improve DO proxy modeling using the same clean matchup condition used in the Turbidity within-1-day experiment.

Main setup:
- Dataset: relaxed v4 tabular dataset
- Target: `DO_mg_l_clean`
- Match condition: `abs_date_diff_days <= 1`
- Train years: BE 2562-2566
- Test years: BE 2567-2568
- Validation: temporal holdout only in this notebook
- Models: small-data linear models, robust linear models, PLS, tree ensembles, XGBoost, and CatBoost
- No DummyMedian baseline in this notebook

Important interpretation: DO is not directly measured by Sentinel-2. This notebook treats DO as an indirect/proxy prediction task.

In [ ]:
# Imports
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from sklearn.exceptions import ConvergenceWarning
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, BayesianRidge, HuberRegressor
from sklearn.svm import LinearSVR
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    XGBOOST_IMPORT_ERROR = exc

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception as exc:
    HAS_CATBOOST = False
    CATBOOST_IMPORT_ERROR = exc

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

print("XGBoost available:", HAS_XGBOOST)
print("CatBoost available:", HAS_CATBOOST)

## 1. Configuration

In [ ]:
# Config
PROJECT_ROOT = Path(r".")
DATA_PATH = PROJECT_ROOT / "Data" / "tabular_data_14" / "train" / "v4" / "rswq_model_input_relaxed_s2_context_2562_2568.csv"

TARGET_COL = "DO_mg_l_clean"
STATION_COL = "station_canonical"
YEAR_COL = "year_be"
MATCH_DAYS_COL = "abs_date_diff_days"

TRAIN_YEAR_MIN = 2562
TRAIN_YEAR_MAX = 2566
TEST_YEAR_MIN = 2567
TEST_YEAR_MAX = 2568
MAX_MATCH_DAYS = 1
RANDOM_STATE = 42

# Use the same curated S2/context setup as the Turbidity within-1-day comparison.
S2_SELECTED = [
    "B2_median",
    "B3_median",
    "B4_median",
    "B5_median",
    "B6_median",
    "B7_median",
    "B8_median",
    "B8A_median",
    "B11_median",
    "B12_median",
]
# S2_SELECTED = [
#     "B1_median", "B2_median", "B5_median",
#     "B9_median", "B12_median",
#     "NDCI_median","ph_formula_s2_script_median","B5_B4_median","AWEIsh_median"
# ]

CONTEXT_SELECTED = [
    "wc_shrub_frac_5000m",
    "wc_bare_frac_1000m",
    "month",
    "wc_tree_frac_1000m",
    "wc_shrub_frac_1000m",
    "day_of_year",
    "wc_crop_frac_1000m",
    "wc_grass_frac_1000m",
    "rain_gsmap_prior_720h_mm",
    "rainy_hours_gsmap_prior_336h",
    "wc_tree_frac_500m",
    "wc_wetland_frac_5000m",
    "jrc_occurrence_mean_500m",
    "era5h_wind_max_prior_168h_m_s",
    "rain_gsmap_prior_336h_mm",
    "era5h_solar_prior_168h_MJ_m2",
    "era5h_air_temp_prior_72h_C",
    "wc_shrub_frac_500m",
    "wc_bare_frac_500m",
    "jrc_recurrence_mean_500m"
]

FEATURE_SETS = {
    "S2 selected": S2_SELECTED,
    "Context selected": CONTEXT_SELECTED,
    "S2 selected + Context selected": S2_SELECTED + CONTEXT_SELECTED,
}

print("Dataset:", DATA_PATH)
print("Target:", TARGET_COL)
print("Train years:", TRAIN_YEAR_MIN, "-", TRAIN_YEAR_MAX)
print("Test years:", TEST_YEAR_MIN, "-", TEST_YEAR_MAX)
print("Match window: abs_date_diff_days <=", MAX_MATCH_DAYS)

## 2. Load Data and Apply Within-1-Day Filter

In [ ]:
# Load data and apply the same clean temporal-match condition as the Turbidity notebook.
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Raw shape:", df_raw.shape)

required_cols = [TARGET_COL, YEAR_COL, MATCH_DAYS_COL] + sorted(set(sum(FEATURE_SETS.values(), [])))
missing_cols = [c for c in required_cols if c not in df_raw.columns]
if missing_cols:
    raise KeyError(f"Missing required columns: {missing_cols}")

df = df_raw.copy()
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[MATCH_DAYS_COL] = pd.to_numeric(df[MATCH_DAYS_COL], errors="coerce")

base_mask = df[TARGET_COL].notna() & df[YEAR_COL].notna() & df[MATCH_DAYS_COL].le(MAX_MATCH_DAYS)
within1_df = df.loc[base_mask].copy()

train_df = within1_df.loc[within1_df[YEAR_COL].between(TRAIN_YEAR_MIN, TRAIN_YEAR_MAX)].copy()
test_df = within1_df.loc[within1_df[YEAR_COL].between(TEST_YEAR_MIN, TEST_YEAR_MAX)].copy()

split_summary = pd.DataFrame([
    {
        "split": "within1_all",
        "rows": len(within1_df),
        "stations": within1_df[STATION_COL].nunique() if STATION_COL in within1_df.columns else np.nan,
        "target_mean": within1_df[TARGET_COL].mean(),
        "target_median": within1_df[TARGET_COL].median(),
        "target_min": within1_df[TARGET_COL].min(),
        "target_max": within1_df[TARGET_COL].max(),
    },
    {
        "split": f"train_{TRAIN_YEAR_MIN}_{TRAIN_YEAR_MAX}",
        "rows": len(train_df),
        "stations": train_df[STATION_COL].nunique() if STATION_COL in train_df.columns else np.nan,
        "target_mean": train_df[TARGET_COL].mean(),
        "target_median": train_df[TARGET_COL].median(),
        "target_min": train_df[TARGET_COL].min(),
        "target_max": train_df[TARGET_COL].max(),
    },
    {
        "split": f"test_{TEST_YEAR_MIN}_{TEST_YEAR_MAX}",
        "rows": len(test_df),
        "stations": test_df[STATION_COL].nunique() if STATION_COL in test_df.columns else np.nan,
        "target_mean": test_df[TARGET_COL].mean(),
        "target_median": test_df[TARGET_COL].median(),
        "target_min": test_df[TARGET_COL].min(),
        "target_max": test_df[TARGET_COL].max(),
    },
])

display(split_summary.round(4))

print("Train years:", sorted(train_df[YEAR_COL].dropna().astype(int).unique().tolist()))
print("Test years :", sorted(test_df[YEAR_COL].dropna().astype(int).unique().tolist()))

if len(train_df) < 20 or len(test_df) < 5:
    raise ValueError("Too few rows after within-1-day temporal filtering.")

## 3. DO Target Audit

In [ ]:
# DO distribution check. Rows are not removed here; this is only an audit.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(train_df[TARGET_COL].dropna(), bins=18, alpha=0.75, label="train")
axes[0].hist(test_df[TARGET_COL].dropna(), bins=18, alpha=0.75, label="test")
axes[0].set_xlabel("DO (mg/L)")
axes[0].set_ylabel("Count")
axes[0].set_title("DO distribution")
axes[0].legend()
axes[0].grid(True, alpha=0.25)

combined_for_box = [train_df[TARGET_COL].dropna(), test_df[TARGET_COL].dropna()]
axes[1].boxplot(combined_for_box, tick_labels=["train", "test"], showmeans=True)
axes[1].set_ylabel("DO (mg/L)")
axes[1].set_title("Train/test target range")
axes[1].grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

print("Potential low-DO / suspicious rows. Not filtered in this notebook.")
audit_cols = [c for c in [
    "sample_id", STATION_COL, YEAR_COL, "field_date", "sat_date", TARGET_COL,
    "match_window", MATCH_DAYS_COL, "valid_water_pixel_count", "water_pixel_fraction", "local_cloud_pct",
    "rain_gsmap_prior_24h_mm", "era5h_runoff_prior_24h_mm",
] if c in within1_df.columns]

display(within1_df.loc[within1_df[TARGET_COL].le(2), audit_cols].sort_values(TARGET_COL).round(4))

## 4. Feature Sets

In [ ]:
# Feature-set audit.
feature_summary = pd.DataFrame({
    "feature_set": list(FEATURE_SETS.keys()),
    "n_features": [len(v) for v in FEATURE_SETS.values()],
    "features": [", ".join(v) for v in FEATURE_SETS.values()],
})
display(feature_summary)

missing_rows = []
for feature_set_name, feature_cols in FEATURE_SETS.items():
    for c in feature_cols:
        missing_rows.append({
            "feature_set": feature_set_name,
            "feature": c,
            "missing_train_pct": train_df[c].isna().mean() * 100,
            "missing_test_pct": test_df[c].isna().mean() * 100,
        })
missing_df = pd.DataFrame(missing_rows).sort_values(["missing_train_pct", "missing_test_pct"], ascending=False)
print("Feature missingness")
display(missing_df.round(2))

## 5. Models and Metrics

In [ ]:
# Metrics and model helpers.
def safe_corr(corr_func, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    if ok.sum() < 3 or np.unique(y_true[ok]).size < 2 or np.unique(y_pred[ok]).size < 2:
        return np.nan
    return float(corr_func(y_true[ok], y_pred[ok])[0])


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ok = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[ok]
    y_pred = y_pred[ok]
    if len(y_true) == 0:
        return {
            "n_eval": 0,
            "r2": np.nan,
            "rmse": np.nan,
            "mae": np.nan,
            "pearson_r": np.nan,
            "spearman_r": np.nan,
            "bias": np.nan,
            "pred_min": np.nan,
            "pred_max": np.nan,
        }
    return {
        "n_eval": int(len(y_true)),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) >= 2 and np.unique(y_true).size > 1 else np.nan,
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "pearson_r": safe_corr(pearsonr, y_true, y_pred),
        "spearman_r": safe_corr(spearmanr, y_true, y_pred),
        "bias": float(np.mean(y_pred - y_true)),
        "pred_min": float(np.min(y_pred)),
        "pred_max": float(np.max(y_pred)),
    }


def linear_pipe(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


def tree_pipe(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])


def make_models(n_features, random_state=42):
    models = {
        "Ridge": linear_pipe(Ridge(alpha=1.0)),
        "ElasticNet": linear_pipe(ElasticNet(alpha=0.03, l1_ratio=0.25, max_iter=20000, random_state=random_state)),
        "BayesianRidge": linear_pipe(BayesianRidge()),
        "HuberRegressor": linear_pipe(HuberRegressor(epsilon=1.35, alpha=0.0001, max_iter=2000)),
        "RandomForest": tree_pipe(RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        "ExtraTrees": tree_pipe(ExtraTreesRegressor(n_estimators=500, min_samples_leaf=2, random_state=random_state, n_jobs=-1)),
        "HistGradientBoosting": tree_pipe(HistGradientBoostingRegressor(max_iter=300, learning_rate=0.03, l2_regularization=0.1, random_state=random_state)),
    }

    for c in [0.3, 0.5, 1.0, 2.0]:
        for epsilon in [0.0, 0.05, 0.10]:
            models[f"LinearSVR_C{c}_eps{epsilon}"] = linear_pipe(
                LinearSVR(C=c, epsilon=epsilon, random_state=random_state, max_iter=30000)
            )

    for n_components in [2, 4, 8]:
        if n_components <= n_features:
            models[f"PLS_{n_components}"] = linear_pipe(PLSRegression(n_components=n_components))

    if HAS_XGBOOST:
        models["XGBoost"] = tree_pipe(XGBRegressor(
            n_estimators=500,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=3.0,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
        ))

    if HAS_CATBOOST:
        models["CatBoost"] = tree_pipe(CatBoostRegressor(
            iterations=600,
            depth=3,
            learning_rate=0.03,
            loss_function="RMSE",
            random_seed=random_state,
            verbose=False,
            allow_writing_files=False,
        ))

    return models


def fit_and_evaluate_model(model, train_data, test_data, feature_cols):
    X_train = train_data[feature_cols].copy()
    X_test = test_data[feature_cols].copy()
    y_train = train_data[TARGET_COL].astype(float).values
    y_test = test_data[TARGET_COL].astype(float).values

    model.fit(X_train, y_train)
    pred_train = np.asarray(model.predict(X_train), dtype=float).reshape(-1)
    pred_test = np.asarray(model.predict(X_test), dtype=float).reshape(-1)

    train_metrics = regression_metrics(y_train, pred_train)
    test_metrics = regression_metrics(y_test, pred_test)

    pred_df = pd.DataFrame({
        "sample_id": test_data["sample_id"].values if "sample_id" in test_data.columns else test_data.index.values,
        STATION_COL: test_data[STATION_COL].values if STATION_COL in test_data.columns else np.nan,
        YEAR_COL: test_data[YEAR_COL].values,
        "observed": y_test,
        "predicted": pred_test,
        "residual": pred_test - y_test,
    })
    pred_df["abs_residual"] = pred_df["residual"].abs()
    return model, train_metrics, test_metrics, pred_df

print("No DummyMedian model is included by design.")

## 6. Run Temporal Model Comparison

In [ ]:
# Train/evaluate all feature-set x model combinations on the temporal split.
rows = []
prediction_tables = {}
fitted_models = {}

for feature_set_name, feature_cols in FEATURE_SETS.items():
    models = make_models(n_features=len(feature_cols), random_state=RANDOM_STATE)
    print(f"Feature set: {feature_set_name} | n_features={len(feature_cols)} | models={len(models)}")

    for model_name, model in models.items():
        try:
            fitted, train_metrics, test_metrics, pred_df = fit_and_evaluate_model(
                model=model,
                train_data=train_df,
                test_data=test_df,
                feature_cols=feature_cols,
            )
            row = {
                "feature_set": feature_set_name,
                "model": model_name,
                "n_features": len(feature_cols),
                "n_train": len(train_df),
                "n_test": len(test_df),
            }
            row.update({f"train_{k}": v for k, v in train_metrics.items()})
            row.update({f"test_{k}": v for k, v in test_metrics.items()})
            rows.append(row)
            prediction_tables[(feature_set_name, model_name)] = pred_df
            fitted_models[(feature_set_name, model_name)] = fitted
        except Exception as exc:
            rows.append({
                "feature_set": feature_set_name,
                "model": model_name,
                "n_features": len(feature_cols),
                "n_train": len(train_df),
                "n_test": len(test_df),
                "error": repr(exc),
            })
            print("ERROR", feature_set_name, model_name, repr(exc))

results = pd.DataFrame(rows)
if "error" in results.columns:
    error_results = results.loc[results["error"].notna()].copy()
    results = results.loc[results["error"].isna()].copy()
else:
    error_results = pd.DataFrame()

results = results.sort_values(["test_r2", "test_rmse"], ascending=[False, True]).reset_index(drop=True)

show_cols = [
    "feature_set", "model", "n_features", "n_train", "n_test",
    "test_r2", "test_rmse", "test_mae", "test_spearman_r", "test_pearson_r", "test_bias",
    "test_pred_min", "test_pred_max", "train_r2", "train_rmse",
]

print("Best DO temporal models, sorted by test R2")
display(results[show_cols].round(4))

if not error_results.empty:
    print("Model errors")
    display(error_results)

## 7. Best Model by Feature Set

In [ ]:
# Best model per feature set.
best_by_feature = (
    results.sort_values(["feature_set", "test_r2", "test_rmse"], ascending=[True, False, True])
    .groupby("feature_set", as_index=False)
    .head(1)
    .sort_values(["test_r2", "test_rmse"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Best DO model by feature set")
display(best_by_feature[show_cols].round(4))

best_overall = results.iloc[0].to_dict()
print("Best overall config")
display(pd.DataFrame([best_overall])[show_cols].round(4))

## 8. Prediction Plots

In [ ]:
# Plot observed vs predicted for the best overall model and best model per feature set.
def plot_observed_predicted(pred_df, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    y_true = pred_df["observed"].to_numpy(dtype=float)
    y_pred = pred_df["predicted"].to_numpy(dtype=float)
    lim_min = min(np.nanmin(y_true), np.nanmin(y_pred)) - 0.25
    lim_max = max(np.nanmax(y_true), np.nanmax(y_pred)) + 0.25

    axes[0].scatter(y_true, y_pred, alpha=0.85)
    axes[0].plot([lim_min, lim_max], [lim_min, lim_max], linestyle="--", color="gray", linewidth=1)
    axes[0].set_xlim(lim_min, lim_max)
    axes[0].set_ylim(lim_min, lim_max)
    axes[0].set_xlabel("Observed DO (mg/L)")
    axes[0].set_ylabel("Predicted DO (mg/L)")
    axes[0].set_title(title)
    axes[0].grid(True, alpha=0.3)

    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].scatter(y_true, pred_df["residual"], alpha=0.85)
    axes[1].set_xlabel("Observed DO (mg/L)")
    axes[1].set_ylabel("Residual: predicted - observed")
    axes[1].set_title("Residuals")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

for _, row in best_by_feature.iterrows():
    key = (row["feature_set"], row["model"])
    pred_df = prediction_tables[key]
    title = f"{row['feature_set']} | {row['model']}\nR2={row['test_r2']:.3f}, RMSE={row['test_rmse']:.3f}"
    print(title)
    display(pred_df.sort_values("abs_residual", ascending=False).head(10).round(4))
    plot_observed_predicted(pred_df, title)

## 9. Feature Importance / Coefficients for Best Overall Model

In [ ]:
# Inspect feature importance for the best overall model.
best_key = (best_overall["feature_set"], best_overall["model"])
best_model = fitted_models[best_key]
best_features = FEATURE_SETS[best_overall["feature_set"]]
final_estimator = best_model.named_steps["model"]

importance = None
importance_label = None

if hasattr(final_estimator, "feature_importances_"):
    importance = np.asarray(final_estimator.feature_importances_, dtype=float)
    importance_label = "feature_importance"
elif hasattr(final_estimator, "get_feature_importance"):
    importance = np.asarray(final_estimator.get_feature_importance(), dtype=float)
    importance_label = "feature_importance"
elif hasattr(final_estimator, "coef_"):
    importance = np.asarray(final_estimator.coef_, dtype=float).reshape(-1)
    if len(importance) != len(best_features):
        importance = np.resize(importance, len(best_features))
    importance_label = "coefficient"

if importance is None or len(importance) == 0:
    print(f"No native feature importance/coefficient available for {best_overall['model']}.")
else:
    if len(importance) != len(best_features):
        importance = np.resize(importance, len(best_features))

    imp_df = pd.DataFrame({
        "feature": best_features,
        importance_label: importance,
        "abs_value": np.abs(importance),
    }).sort_values("abs_value", ascending=False).reset_index(drop=True)

    print(f"Best overall model: {best_overall['feature_set']} | {best_overall['model']}")
    display(imp_df.head(30).round(6))

    fig, ax = plt.subplots(figsize=(8, 6))
    plot_df = imp_df.head(20).iloc[::-1]
    ax.barh(plot_df["feature"], plot_df[importance_label])
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel(importance_label)
    ax.set_title(f"Top features | {best_overall['model']} | {best_overall['feature_set']}")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

## SHAP Global Importance by Feature Set

This section computes model-agnostic SHAP global importance for the best DO model in each feature set:

- `S2 only`: best model selected from Sentinel-2 features only
- `Context only`: best model selected from context features only
- `S2 + Context`: best model selected from Sentinel-2 plus context features

SHAP values are computed on the temporal test set using the training set as the background. The target is DO in mg/L. Because DO is not directly optically retrieved by Sentinel-2, these explanations should be interpreted as proxy-model explanations rather than direct physical retrieval mechanisms.


In [ ]:
# SHAP global importance for the best DO model in each feature set
import shap

DO_FEATURE_SET_DISPLAY = {
    "S2 selected": "S2 only",
    "Context selected": "Context only",
    "S2 selected + Context selected": "S2 + Context",
}

def do_feature_group(feature_name):
    if feature_name in S2_SELECTED:
        return "S2"
    if feature_name in CONTEXT_SELECTED:
        return "Context"
    return "Other"

DO_GROUP_COLORS = {
    "S2": "#377eb8",
    "Context": "#4daf4a",
    "Other": "#999999",
}

def normalize_abs_importance(values):
    values = np.asarray(values, dtype=float)
    total = np.nansum(np.abs(values))
    if total == 0 or not np.isfinite(total):
        return np.zeros_like(values, dtype=float)
    return np.abs(values) / total

def make_pipeline_predict_fn(model, feature_cols):
    def predict_fn(X):
        X_df = pd.DataFrame(X, columns=feature_cols)
        return np.asarray(model.predict(X_df), dtype=float).reshape(-1)
    return predict_fn

SHAP_BACKGROUND_SIZE = min(50, len(train_df))
SHAP_MAX_DISPLAY = 15

do_shap_global_tables = {}
do_shap_values_by_feature_set = {}
do_shap_model_summary_rows = []

for _, best_row in best_by_feature.sort_values("feature_set").iterrows():
    feature_set_name = best_row["feature_set"]
    display_name = DO_FEATURE_SET_DISPLAY.get(feature_set_name, feature_set_name)
    model_name = best_row["model"]
    feature_cols = list(FEATURE_SETS[feature_set_name])
    model = fitted_models[(feature_set_name, model_name)]

    X_background = train_df[feature_cols].copy().sample(
        n=SHAP_BACKGROUND_SIZE,
        random_state=RANDOM_STATE,
    )
    X_explain = test_df[feature_cols].copy()
    predict_fn = make_pipeline_predict_fn(model, feature_cols)

    explainer = shap.Explainer(
        predict_fn,
        X_background,
        algorithm="permutation",
    )
    shap_values = explainer(
        X_explain,
        max_evals=max(2 * len(feature_cols) + 1, 200),
    )

    shap_array = np.asarray(shap_values.values, dtype=float)
    if shap_array.ndim == 3:
        shap_array = shap_array[:, :, 0]

    shap_global_df = pd.DataFrame({
        "feature_set": display_name,
        "model": model_name,
        "feature": feature_cols,
        "feature_group": [do_feature_group(c) for c in feature_cols],
        "mean_abs_shap_do_mg_l": np.nanmean(np.abs(shap_array), axis=0),
        "mean_shap_do_mg_l": np.nanmean(shap_array, axis=0),
    })
    shap_global_df["shap_importance_norm"] = normalize_abs_importance(
        shap_global_df["mean_abs_shap_do_mg_l"]
    )
    shap_global_df["average_direction"] = np.where(
        shap_global_df["mean_shap_do_mg_l"] >= 0,
        "pushes predicted DO up on average",
        "pushes predicted DO down on average",
    )
    shap_global_df = shap_global_df.sort_values(
        "mean_abs_shap_do_mg_l",
        ascending=False,
    ).reset_index(drop=True)

    do_shap_global_tables[display_name] = shap_global_df
    do_shap_values_by_feature_set[display_name] = shap_values

    do_shap_model_summary_rows.append({
        "feature_set": display_name,
        "model": model_name,
        "n_features": len(feature_cols),
        "n_train": int(best_row["n_train"]),
        "n_test": int(best_row["n_test"]),
        "test_r2": best_row["test_r2"],
        "test_rmse": best_row["test_rmse"],
        "test_mae": best_row["test_mae"],
        "test_spearman_r": best_row["test_spearman_r"],
        "test_bias": best_row["test_bias"],
        "top_shap_feature": shap_global_df.loc[0, "feature"],
        "top_mean_abs_shap": shap_global_df.loc[0, "mean_abs_shap_do_mg_l"],
    })

    print(f"DO SHAP global importance | {display_name} | {model_name}")
    display(shap_global_df.round(6))

    fig, ax = plt.subplots(figsize=(8, 6))
    plot_df = shap_global_df.head(SHAP_MAX_DISPLAY).iloc[::-1]
    ax.barh(
        plot_df["feature"],
        plot_df["mean_abs_shap_do_mg_l"],
        color=plot_df["feature_group"].map(DO_GROUP_COLORS),
    )
    ax.set_xlabel("Mean absolute SHAP value for predicted DO (mg/L)")
    ax.set_title(f"DO SHAP global importance | {display_name} | {model_name}")
    ax.grid(True, axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

do_shap_model_summary = pd.DataFrame(do_shap_model_summary_rows).sort_values(
    ["test_r2", "test_rmse"],
    ascending=[False, True],
).reset_index(drop=True)

print("DO SHAP model summary")
display(do_shap_model_summary.round(4))

do_shap_top10_comparison = pd.concat(
    [df.head(10).assign(rank=np.arange(1, min(10, len(df)) + 1)) for df in do_shap_global_tables.values()],
    ignore_index=True,
)[[
    "feature_set", "rank", "model", "feature", "feature_group",
    "mean_abs_shap_do_mg_l", "mean_shap_do_mg_l", "shap_importance_norm",
    "average_direction",
]]

print("Top 10 DO SHAP features by feature set")
display(do_shap_top10_comparison.round(6))


## 10. Interpretation Notes

- This is a first temporal diagnostic for DO under the stricter within-1-day Sentinel-2 matchup condition.
- DO is a proxy target for remote sensing. A strong result should be interpreted as indirect prediction from environmental context and water-surface/landscape proxies, not direct optical retrieval.
- This notebook intentionally does not choose a final paper-safe model. The next step should be robust validation using leave-one-year-out, station-wise validation, repeated random splits, and bootstrap confidence intervals for the best candidate configurations.

## Reporting Summary: Requested Model Families

This table compares the requested model families on the same within-1-day temporal holdout: train years BE 2562-2566 and test years BE 2567-2568. For PLS and LinearSVR, the best component count or `C`/`epsilon` configuration within each feature set is retained. It is an exploratory comparison because the same holdout was used to screen configurations.


In [ ]:
# Reporting table: one representative result per requested model family and feature set.
def do_report_model_family(model_name):
    model_name = str(model_name)
    if model_name.startswith("LinearSVR_"):
        return "LinearSVR"
    if model_name.startswith("PLS_"):
        return "PLS"
    return model_name


REPORT_MODEL_FAMILIES = [
    "Ridge",
    "ElasticNet",
    "HuberRegressor",
    "PLS",
    "LinearSVR",
    "RandomForest",
    "ExtraTrees",
    "XGBoost",
    "CatBoost",
]

do_report_candidates = results.copy()
do_report_candidates["model_family"] = do_report_candidates["model"].map(do_report_model_family)
do_report_candidates = do_report_candidates.loc[
    do_report_candidates["model_family"].isin(REPORT_MODEL_FAMILIES)
].copy()

do_report_selected = (
    do_report_candidates
    .sort_values(
        ["feature_set", "model_family", "test_r2", "test_rmse"],
        ascending=[True, True, False, True],
    )
    .groupby(["feature_set", "model_family"], as_index=False)
    .head(1)
    .copy()
)

do_model_summary = (
    do_report_selected
    .rename(columns={
        "feature_set": "feature set",
        "test_r2": "R2",
        "test_rmse": "RMSE",
        "test_mae": "MAE",
        "test_spearman_r": "spearman r",
        "test_bias": "bias",
    })
    [["model", "feature set", "R2", "RMSE", "MAE", "spearman r", "bias"]]
    .sort_values(["R2", "RMSE"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Within-1-day temporal test: train 2562-2566, test 2567-2568.")
display(do_model_summary.round(4))

print("Best model by test R2")
display(do_model_summary.iloc[[0]].round(4))


## Locked model artifact export

This cell serializes the fixed model configuration used for mapping. Notebook 04 must load these artifacts and must not refit them.

In [ ]:
# Export the locked DO S2+Context CatBoost variant for Notebook 04.
from pathlib import Path
import joblib

ARTIFACT_DIR = Path(r".\Models\mapping_v1\artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

feature_set_name = "S2 selected + Context selected"
feature_names = FEATURE_SETS[feature_set_name]
model_key = (feature_set_name, "CatBoost")

if "fitted_models" in globals() and model_key in fitted_models:
    estimator = fitted_models[model_key]
else:
    estimator = make_models(len(feature_names), RANDOM_STATE)["CatBoost"]
    estimator.fit(train_df[feature_names], train_df[TARGET_COL])

bundle = {
    "artifact_version": "1.0.0",
    "artifact_id": "do_s2_context",
    "target": "DO",
    "target_column": TARGET_COL,
    "feature_set": "S2 + Context",
    "feature_names": list(feature_names),
    "estimator": estimator,
    "prediction_contract": {
        "strategy": "raw_regression",
        "inverse_transform": "identity",
        "unit": "mg/L",
        "interpretation": "indirect proxy prediction",
    },
    "split_contract": {
        "match_window_days": MAX_MATCH_DAYS,
        "train_year_be": [TRAIN_YEAR_MIN, TRAIN_YEAR_MAX],
        "test_year_be": [TEST_YEAR_MIN, TEST_YEAR_MAX],
    },
}

output_path = ARTIFACT_DIR / "do_s2_context_catboost.joblib"
joblib.dump(bundle, output_path, compress=3)
display(pd.DataFrame([{"artifact_id": bundle["artifact_id"], "path": output_path}]))